In [9]:
!pip install ucimlrepo

In [10]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

ds = fetch_ucirepo(id=484)
df = ds.data.features.copy()
print(ds.variables)
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isna().sum())

           name     role         type demographic  \
0       User ID       ID  Categorical        None   
1    Category 1  Feature   Continuous        None   
2    Category 2  Feature   Continuous        None   
3    Category 3  Feature   Continuous        None   
4    Category 4  Feature   Continuous        None   
5    Category 5  Feature   Continuous        None   
6    Category 6  Feature   Continuous        None   
7    Category 7  Feature   Continuous        None   
8    Category 8  Feature   Continuous        None   
9    Category 9  Feature   Continuous        None   
10  Category 10  Feature   Continuous        None   

                                        description units missing_values  
0                                              None  None             no  
1            Average user feedback on art galleries  None             no  
2              Average user feedback on dance clubs  None             no  
3               Average user feedback on juice bars  None      

In [11]:
df = df.drop(columns=[c for c in df.columns if 'user' in str(c).lower()], errors='ignore')
df = df.dropna(axis=1, how='all')

print("Ausentes:", df.isna().sum().sum())
df = df.dropna()

df = df.select_dtypes(include=[np.number])

nomes = ['galerias_arte','casas_noturnas','bares_suco','restaurantes','museus',
         'resorts','parques','praias','teatros','instituicoes_religiosas']
if df.shape[1] == len(nomes):
    df.columns = nomes

X = df.values
n, d = X.shape
print(f"Matriz X final: n = {n}, d = {d}")

scaler = StandardScaler()
Z = scaler.fit_transform(X)
print("ANTES  (linha 0):", np.round(X[0], 3))
print("DEPOIS (linha 0):", np.round(Z[0], 3))
print("média por coluna ≈ 0:", np.round(Z.mean(axis=0), 3))
print("desvio por coluna ≈ 1:", np.round(Z.std(axis=0), 3))

Ausentes: 0
Matriz X final: n = 980, d = 10
ANTES  (linha 0): [0.93 1.8  2.29 0.62 0.8  2.42 3.19 2.79 1.82 2.42]
DEPOIS (linha 0): [ 0.113  0.936  1.62   0.313 -0.32   1.07   1.159 -0.328  0.688 -1.181]
média por coluna ≈ 0: [-0.  0. -0.  0. -0.  0. -0. -0. -0. -0.]
desvio por coluna ≈ 1: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
Ks = range(1, 11)
J, S = {}, {}
for k in Ks:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42).fit(Z)
    J[k] = km.inertia_
    if k >= 2:
        S[k] = silhouette_score(Z, km.labels_)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(J), list(J.values()), 'o-'); ax[0].set(title='Curva do cotovelo', xlabel='k', ylabel='J(k)')
ax[1].plot(list(S), list(S.values()), 'o-'); ax[1].set(title='Silhueta média', xlabel='k', ylabel='s̄')
plt.tight_layout(); plt.show()
print(pd.DataFrame({'J': J, 'silhueta': S}))